# Select ranked TMDB candidates
Replay the saved TMDB responses in vote order. For a live demonstration, set `online_demo = True` and provide a local `TMDB_READ_TOKEN`. The demonstration requests the first 1970s page with the same filters.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import requests
import pandas as pd
import json
import time
import hashlib
import os

project_dir = Path.cwd()
temp = TemporaryDirectory(prefix="ads206_")
# Extract only the candidate cache.
with ZipFile("data_compact.zip") as data:
    for name in data.namelist():
        if name.startswith("01_raw/tmdb_candidates/") or name == "02_audit/tmdb_candidates.json":
            data.extract(name, temp.name)
folder = Path(temp.name)
base = folder / "01_raw/tmdb_candidates"
run = base / (base / "latest_run.txt").read_text(encoding="utf-8").strip()
cache_index = json.loads((run / "cache_index.json").read_text(encoding="utf-8"))
saved_pool = json.loads((folder / "02_audit/tmdb_candidates.json").read_text(encoding="utf-8"))
# Apply the same cutoffs in every decade.
pages_to_read = saved_pool["pages_per_decade"]
rating_min, votes_min = 7.0, 1000
live, online_demo = False, False
settings = dict(os.environ)
if (project_dir / ".env").exists():
    for line in (project_dir / ".env").read_text(encoding="utf-8-sig").splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            key, value = line.split("=", 1)
            settings[key.strip()] = value.strip().strip("\"'")
headers = {"User-Agent": "ADS206-film-study/1.0 (educational)", "Accept": "application/json"}
if live or online_demo:
    assert settings.get("TMDB_READ_TOKEN"), "Set TMDB_READ_TOKEN locally, or set online_demo = False for offline use"
    headers["Authorization"] = "Bearer " + settings["TMDB_READ_TOKEN"]


## 1. Request and save

`requests.get` sends the web request. Keep each response and its log. Pause between requests; retry temporary failures up to three times. Headers and the token are never logged.

If your network needs a proxy, the optional `HTTPS_PROXY` line in `.env` supplies it for these requests. No system setting is changed.
The saved run pointer locates the raw replies. Optional live mode requests only missing responses and records each request.


In [2]:
from email.utils import parsedate_to_datetime

def retry_delay(reply, attempt):
    delay = 2 ** attempt
    value = reply.headers.get("Retry-After", "").strip() if reply is not None else ""
    if value.isdecimal():
        delay = max(delay, int(value))
    elif value:
        try:
            until = parsedate_to_datetime(value)
            delay = max(delay, (until - datetime.now(timezone.utc)).total_seconds())
        except (TypeError, ValueError, OverflowError):
            pass  # Malformed header: use the normal backoff.
    if delay > 60:
        raise RuntimeError("Server asked for a longer wait. Stop this run and retry later.")
    return delay

# Fetch API data with caching and error handling
def fetch(params, name, url="https://api.themoviedb.org/3/discover/movie", force=False):
    cached = run / cache_index.get(name, name)
    if not force and cached.exists():
        try:
            return json.loads(cached.read_bytes())
        except (ValueError, UnicodeError) as error:
            raise ValueError(f"Invalid cached JSON: {cached.name}") from error
    assert live or force, "Saved API response missing"
    target = project_dir / "01_raw/tmdb_candidates" / datetime.now(timezone.utc).strftime("live_%Y%m%dT%H%M%S%fZ")
    target.mkdir(parents=True, exist_ok=False)
    for attempt in range(3):
        reply = None
        start = time.perf_counter()
        log = {"time_utc": datetime.now(timezone.utc).isoformat(), "url": url, "params": params, "retry_count": attempt}
        try:
            # Use a proxy if specified in the environment settings
            reply = requests.get(url, params=params, headers=headers, timeout=(5, 15),
                                 proxies={"https": settings["HTTPS_PROXY"]} if settings.get("HTTPS_PROXY") else None)
            assert settings["TMDB_READ_TOKEN"].encode() not in reply.content, "Credential in response"
            file = f"response_{attempt}.json"
            (target / file).write_bytes(reply.content)
            log.update(status=reply.status_code, file=file, sha256=hashlib.sha256(reply.content).hexdigest())
            print("Network HTTP status:", reply.status_code)
            if reply.ok:
                try:
                    parsed = reply.json()
                except ValueError:
                    log["error"] = "invalid JSON"
        except requests.RequestException as error:
            log["error"] = type(error).__name__
        log["seconds"] = time.perf_counter() - start
        with (target / "requests.jsonl").open("a", encoding="utf-8") as output:
            output.write(json.dumps(log) + "\n")
        time.sleep(1.2)
        if reply is not None and reply.ok:
            if "error" in log:
                raise ValueError(f"Invalid JSON for {name}. See {target / 'requests.jsonl'}")
            return parsed
        if reply is not None and reply.status_code < 500 and reply.status_code not in (408, 429):
            reply.raise_for_status()
        if attempt < 2:
            time.sleep(retry_delay(reply, attempt))
    raise ConnectionError(f"API request failed for {name}. See {target / 'requests.jsonl'}")

def candidate_params(decade, page_number):
    return {"primary_release_date.gte": f"{decade}-01-01",
        "primary_release_date.lte": f"{decade + 9}-12-31",
        "sort_by": "vote_count.desc", "page": page_number, "language": "en-US",
        "include_adult": "false", "include_video": "false",
        "vote_average.gte": rating_min, "vote_count.gte": votes_min}

if online_demo:
    demo = fetch(candidate_params(1970, 1), "online_demo.json", force=True)
    print("Live page:", demo["page"], "Returned films:", len(demo["results"]))
    if demo["results"]:
        print("First film:", demo["results"][0]["title"])


## 2. Read candidate pages

Use primary release dates, rating >= 7.0, vote count >= 1,000 and `vote_count.desc`. Read the recorded pages for each decade. Extra pages continue the same vote-ranked order when replacements are needed. No country or genre cutoff is applied. English labels do not restrict production country.

Only returned records count as collected data. The API-reported totals are saved but not treated as verified coverage estimates. This vote-ranked list is a purposive candidate frame, not a random sample.

In [3]:
records, seen_ids = [], set()
# go through five decades
for decade in range(1970, 2020, 10):
    for page_number in range(1, pages_to_read[f"{decade}s"] + 1):
        params = candidate_params(decade, page_number)
        name = f"discover_{decade}.json" if page_number == 1 else f"discover_{decade}_page{page_number}.json"
        data = fetch(params, name)
        assert data["page"] == page_number and isinstance(data["results"], list)
        rows = data["results"]
        assert len({row["id"] for row in rows}) == len(rows), "Repeated ID on this page."
        #check that the release dates are within the decade and that the rating fields are of the correct type
        assert all(decade <= int(row["release_date"][:4]) < decade + 10 for row in rows), "Check dates."
        assert all(isinstance(row["vote_count"], int) and isinstance(row["vote_average"], (int, float))
                   for row in rows), "Check rating fields."
        assert all(row["vote_average"] >= rating_min and row["vote_count"] >= votes_min for row in rows)
        assert all(a["vote_count"] >= b["vote_count"] for a, b in zip(rows, rows[1:])), "Check sorting."
        for rank, row in enumerate(rows, 1):
            # A film can appear on two pages.
            if row["id"] in seen_ids:
                continue
            seen_ids.add(row["id"])
            records.append({"tmdb_id": row["id"], "title": row["title"],
                "original_title": row["original_title"], "release_date": row["release_date"],
                "decade": f"{decade}s", "page": page_number, "rank_on_page": rank,
                "rating": row["vote_average"], "vote_count": row["vote_count"],
                "original_language": row["original_language"], "genre_ids": row["genre_ids"],
                "overview": row.get("overview", ""), "selected_for_main_analysis": False,
                "tmdb_url": f"https://www.themoviedb.org/movie/{row['id']}"})
assert len({row["tmdb_id"] for row in records}) == len(records), "Repeated ID across decades."


## 3. Read film details

Use each TMDB ID to request IMDb ID, runtime, all production countries and genre names. Keep the full response. These fields support later identity and full-film checks; they do not prove that a Cinemetrics measurement covers the whole film.

The credits endpoint supplies crew entries. Keep Director names for later title/year/director checks.

New live requests append credits to the movie response to reduce requests. Earlier saved runs used a separate credits response; replay supports both formats.

In [4]:
# Read the details for each film, including credits, and update the records with additional information.
for row in records:
    uid = row["tmdb_id"]
    item = fetch({"language": "en-US", "append_to_response": "credits"}, f"movie_{uid}.json",
                 f"https://api.themoviedb.org/3/movie/{uid}")
    assert item["id"] == uid
    # Update the record with additional details from the API response, including IMDb ID, runtime, production countries, genres, title, release date, rating, vote count, and overview.
    row.update(imdb_id=item.get("imdb_id"), runtime_minutes=item.get("runtime"),
               countries=[country["name"] for country in item.get("production_countries", [])],
               country_codes=[country["iso_3166_1"] for country in item.get("production_countries", [])],
               genres=[genre["name"] for genre in item.get("genres", [])],
               title_details=item.get("title"), release_date_details=item.get("release_date"),
               rating_details=item.get("vote_average"), vote_count_details=item.get("vote_count"),
               overview_details=item.get("overview", ""),
               )
    credits = item.get("credits")
    # The archive includes both appended and separate credits.
    if credits is None:
        credits = fetch({"language": "en-US"}, f"credits_{uid}.json",
                        f"https://api.themoviedb.org/3/movie/{uid}/credits")
    assert credits.get("id", uid) == uid
    # Update the record with the names and IDs of directors from the credits information.
    row["directors"] = [person["name"] for person in credits.get("crew", []) if person.get("job") == "Director"]
    row["director_ids"] = [person["id"] for person in credits.get("crew", []) if person.get("job") == "Director"]
print("Film details read:", len(records))

Film details read: 260


## 4. Apply the collection standard

| Check | Rule |
| --- | --- |
| Dates | Primary release date from 1970-01-01 to 2019-12-31 |
| Candidate source | TMDB; the same query and ordering for every decade |
| Rating and votes | TMDB rating >= 7.0 and vote count >= 1,000 |
| Page budget | Read the recorded page counts; extend the same order if needed |
| Geography | No country restriction; later record all production countries |
| Film scope | Feature films; exclude TV episodes, shorts, clips and excerpts |
| Identity | Check shared IMDb ID and year; otherwise review title, year and director |
| Measurements | A full-film Cinemetrics measurement is needed for shot-length analysis |
| Duplicates | One film in the final film table; keep repeat measurements for review |
| Missing evidence | Keep the candidate out of the frozen table until a measurement supports it |

Ratings are TMDB audience scores. Vote counts count ratings, not viewers. English labels and original language do not establish production country. A first-page list overrepresents heavily rated films.

Apply the same rules to every decade. Do not choose or change films because of their shot-length results. Numerical cutoffs were confirmed by the student. The working target is 20 passing films per decade.

In [5]:
# Validate that the saved candidate ID order matches the current records and that the vote counts are monotonically decreasing within each decade. Also, check that the number of unique films is as expected. 
assert [r["tmdb_id"] for r in records] == [r["tmdb_id"] for r in saved_pool["records"]]
view = pd.DataFrame(records)
assert view.groupby("decade")["vote_count"].apply(lambda s: s.is_monotonic_decreasing).all()
assert len(view) == view["tmdb_id"].nunique() == 260
print("Saved candidate ID order matches:", len(view))
display(view[["tmdb_id", "decade", "title", "rating", "vote_count"]].head(5))

Saved candidate ID order matches: 260


,tmdb_id,decade,title,rating,vote_count
0,238,1970s,The Godfather,8.689,23952
1,11,1970s,Star Wars,8.200,23146
2,348,1970s,Alien,8.176,17101
3,240,1970s,The Godfather Part II,8.572,14594
4,185,1970s,A Clockwork Orange,8.167,14299


## Credits

<a href="https://www.themoviedb.org"><img src="https://www.themoviedb.org/assets/2/v4/logos/v2/blue_short-8e7b30f73a4020692ccca9c88bafe5dcb6f8a62a4c6bc55cd9ba82bb2cd95f6c.svg" width="120" alt="TMDB"></a>

This product uses the TMDB API but is not endorsed or certified by TMDB.

[TMDB API documentation](https://developer.themoviedb.org/reference/discover-movie) | [TMDB attribution](https://www.themoviedb.org/about/logos-attribution)

I ran the Python collection scripts. Codex helped draft and debug the scripts and check the offline replay. The README records the project-wide use of Codex.